# Part 7c U55C: Validation

Compare QKeras CPU, hls4ml bit-accurate CPU, and U55C hardware outputs for the selected fold/sweep.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

WORKSPACE = Path.cwd()
ML_BASELINE = WORKSPACE.parent
COYOTE_ROOT = Path("/pub/scratch/sdeheredia/Coyote")
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
if str(ML_BASELINE) not in sys.path:
    sys.path.insert(0, str(ML_BASELINE))

DEFAULT_ITERATION_ROOT = WORKSPACE / "artifacts/cnn_small_hls_opt_img512/notebook_pruned_qat/pruned_qat_w6_a6_s50_rf8_0cfa065db9d4"
DEFAULT_FOLD = 0
DEFAULT_HLS_SWEEP = "hls_1e918f3210d4"
ABI = {
    "img_size": 512,
    "pixels_per_sample": 512 * 512,
    "axi_data_bits": 512,
    "fixed_width": 16,
    "fixed_integer": 6,
    "fixed_fraction": 10,
    "pixels_per_beat": 32,
    "beats_per_sample": 8192,
    "input_bytes_per_sample": 512 * 512 * 2,
    "output_bytes_per_sample": 64,
}

def read_json(path: Path) -> dict:
    return json.loads(Path(path).read_text())

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True))

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_tree(root: Path, patterns=(".cpp", ".h", ".hpp", ".svh", ".txt", ".cmake", "CMakeLists.txt")) -> str:
    h = hashlib.sha256()
    root = Path(root)
    if not root.exists():
        return ""
    for path in sorted(p for p in root.rglob("*") if p.is_file()):
        if path.name == "CMakeLists.txt" or any(path.name.endswith(pat) for pat in patterns):
            h.update(str(path.relative_to(root)).encode())
            h.update(path.read_bytes())
    return h.hexdigest()

def run(cmd: list[str], cwd: Path, log_path: Path | None = None) -> subprocess.CompletedProcess:
    print("$", " ".join(map(str, cmd)))
    proc = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text("STDOUT\n" + proc.stdout + "\nSTDERR\n" + proc.stderr)
    if proc.stdout:
        print(proc.stdout[-4000:])
    if proc.stderr:
        print(proc.stderr[-4000:])
    proc.check_returncode()
    return proc

def sigmoid(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))

def clean_rows(csv_path: Path) -> list[dict]:
    if not Path(csv_path).exists():
        return []
    with Path(csv_path).open(newline="") as handle:
        rows = [row for row in csv.DictReader(handle)]
    return [row for row in rows if row.get("sample_index") not in ("sample_index", "", None)]

def binary_loss(label: float, prob: float) -> float:
    p = float(np.clip(prob, 1e-7, 1.0 - 1e-7))
    return float(-(label * np.log(p) + (1.0 - label) * np.log(1.0 - p)))

def rows_from_logits(meta_rows: list[dict], logits: np.ndarray) -> list[dict]:
    probs = sigmoid(logits)
    rows = []
    for i, (meta, logit, prob) in enumerate(zip(meta_rows, logits, probs)):
        label = int(meta["class_label"])
        pred = int(prob >= 0.5)
        rows.append({
            "sample_index": i,
            "sample_id": meta.get("sample_id", ""),
            "app_name": meta.get("app_name", ""),
            "class_label": label,
            "class_name": meta.get("class_name", "standalone" if label else "benign"),
            "ro_count": meta.get("ro_count", ""),
            "bitstream_path": meta.get("bitstream_path", ""),
            "logit": f"{float(logit):.9f}",
            "probability": f"{float(prob):.9f}",
            "predicted_label": pred,
            "correct": int(pred == label),
            "per_sample_bce_loss": f"{binary_loss(label, prob):.9f}",
            "per_sample_log_loss": f"{binary_loss(label, prob):.9f}",
        })
    return rows

def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("")
        return
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


In [ ]:
ITERATION_ROOT = DEFAULT_ITERATION_ROOT
SELECTED_FOLD = DEFAULT_FOLD
REQUESTED_HLS_SWEEP = DEFAULT_HLS_SWEEP

def discover_sweeps(iteration_root: Path, fold: int) -> list[dict]:
    out = []
    for sweep in sorted((iteration_root / "hls_sweeps").glob("*")):
        project = sweep / f"fold_{fold}" / "project"
        conv = project / "conversion_manifest.json"
        if conv.exists() and (project / "firmware").is_dir():
            payload = read_json(conv)
            out.append({
                "name": sweep.name,
                "path": sweep,
                "project": project,
                "project_name": payload["project_name"],
                "hls_fingerprint": payload.get("hls_fingerprint", ""),
            })
    return out

sweeps = discover_sweeps(ITERATION_ROOT, SELECTED_FOLD)
if not sweeps:
    raise FileNotFoundError(f"No generated hls4ml projects found below {ITERATION_ROOT}")
selected = next((s for s in sweeps if s["name"] == REQUESTED_HLS_SWEEP), sweeps[0])

HLS_SWEEP_ROOT = selected["path"]
HLS_PROJECT_DIR = selected["project"]
PROJECT_NAME = selected["project_name"]
FOLD_DIR = ITERATION_ROOT / f"fold_{SELECTED_FOLD}"
PARITY_DIR = HLS_SWEEP_ROOT / f"fold_{SELECTED_FOLD}" / "parity"
U55C_ROOT = HLS_SWEEP_ROOT / f"fold_{SELECTED_FOLD}" / "u55c_deployment"
PREPARED_INPUTS_DIR = U55C_ROOT / "prepared_inputs"
STAGED_HW_DIR = U55C_ROOT / "coyote_hw"
STAGED_SW_DIR = U55C_ROOT / "coyote_sw"
VALIDATION_DIR = HLS_SWEEP_ROOT / f"fold_{SELECTED_FOLD}" / "u55c_validation"

print("iteration root:", ITERATION_ROOT)
print("selected fold:", SELECTED_FOLD)
print("selected sweep:", selected["name"])
print("hls project:", HLS_PROJECT_DIR)
print("artifact root:", U55C_ROOT)
print("available sweeps:", [s["name"] for s in sweeps])


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

try:
    from train import compute_metrics_from_outputs, save_checkpoint_plots
    USING_TRAIN_PLOTS = True
except Exception as exc:
    USING_TRAIN_PLOTS = False
    print("train.py plotting utilities unavailable; using local fallback:", exc)

    def compute_metrics_from_outputs(total_loss, all_labels, all_probs):
        labels = np.asarray(all_labels, dtype=np.float32)
        probs = np.asarray(all_probs, dtype=np.float32)
        pred = (probs >= 0.5).astype(np.int32)
        metrics = {
            "labels": labels,
            "probs": probs,
            "bce_loss": float(total_loss),
            "log_loss": float(total_loss),
            "accuracy": float(accuracy_score(labels, pred)),
            "balanced_accuracy": float(balanced_accuracy_score(labels, pred)),
            "precision": float(precision_score(labels, pred, zero_division=0)),
            "recall": float(recall_score(labels, pred, zero_division=0)),
            "f1": float(0.0),
            "mcc": float(0.0),
            "ece": float("nan"),
            "roc_auc": float(roc_auc_score(labels, probs)),
            "pr_auc": float(average_precision_score(labels, probs)),
            "confusion_matrix": confusion_matrix(labels, pred),
            "reliability_bins": [],
        }
        p = metrics["precision"]
        r = metrics["recall"]
        metrics["f1"] = float(2 * p * r / max(p + r, 1e-12))
        return metrics

    def save_checkpoint_plots(out_dir, checkpoint_label, canonical_metrics, aug_metrics=None, split_info=None, run_params=None):
        labels = canonical_metrics["labels"]
        probs = canonical_metrics["probs"]
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
        for label_value, name in [(0, "benign"), (1, "standalone")]:
            axes[0].hist(probs[labels == label_value], bins=20, range=(0, 1), histtype="step", density=True, label=name)
        fpr, tpr, _ = roc_curve(labels, probs)
        prec, rec, _ = precision_recall_curve(labels, probs)
        axes[1].plot(fpr, tpr, label=f"ROC-AUC={canonical_metrics['roc_auc']:.4f}")
        axes[1].plot([0, 1], [0, 1], "k:", linewidth=1)
        axes[2].plot(rec, prec, label=f"PR-AUC={canonical_metrics['pr_auc']:.4f}")
        axes[0].set_title("Score Histograms")
        axes[1].set_title("ROC")
        axes[2].set_title("Precision-Recall")
        for ax in axes:
            ax.legend(fontsize=8)
        fig.tight_layout()
        path = Path(out_dir) / f"{checkpoint_label}_evaluation_plots.png"
        fig.savefig(path, dpi=160)
        plt.close(fig)
        print("Saved fallback checkpoint plots:", path)

qkeras_rows = clean_rows(PARITY_DIR / "qkeras_per_sample.csv")
hls_rows = clean_rows(PARITY_DIR / "hls_per_sample.csv")
prep_rows = clean_rows(PREPARED_INPUTS_DIR / "manifest.csv")
hw_raw_rows = clean_rows(U55C_ROOT / "hardware_per_sample.csv")
if not qkeras_rows or not hls_rows:
    raise FileNotFoundError(f"Missing parity rows in {PARITY_DIR}")
if not hw_raw_rows:
    raise FileNotFoundError(f"Missing U55C hardware rows: {U55C_ROOT / 'hardware_per_sample.csv'}")

hw_logits_by_idx = {int(row["sample_index"]): float(row["logit"]) for row in hw_raw_rows}
hw_logits = np.asarray([hw_logits_by_idx[int(row["sample_index"])] for row in prep_rows], dtype=np.float32)
hw_rows = rows_from_logits(prep_rows, hw_logits)
write_csv(U55C_ROOT / "hardware_per_sample_enriched.csv", hw_rows)
np.save(U55C_ROOT / "y_hw.npy", hw_logits)

def metrics_from_rows(rows: list[dict]) -> dict:
    labels = np.asarray([int(row["class_label"]) for row in rows], dtype=np.float32)
    probs = np.asarray([float(row["probability"]) for row in rows], dtype=np.float32)
    losses = np.asarray([float(row["per_sample_bce_loss"]) for row in rows], dtype=np.float32)
    return compute_metrics_from_outputs(float(np.mean(losses)), labels, probs)

stages = {
    "QKeras CPU": qkeras_rows,
    "hls4ml CPU": hls_rows,
    "U55C hardware": hw_rows,
}
summary = {}
for name, rows in stages.items():
    metrics = metrics_from_rows(rows)
    summary[name] = {k: float(metrics[k]) for k in ["accuracy", "balanced_accuracy", "roc_auc", "pr_auc", "bce_loss"]}
    print(f"\n{name}")
    for key, value in summary[name].items():
        print(f"  {key}: {value:.6f}")

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
write_json(VALIDATION_DIR / "comparison_summary.json", summary)


## Comparison Plots


In [ ]:
labels = np.asarray([int(row["class_label"]) for row in qkeras_rows], dtype=np.int32)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for name, rows in stages.items():
    probs = np.asarray([float(row["probability"]) for row in rows], dtype=np.float32)
    fpr, tpr, _ = roc_curve(labels, probs)
    prec, rec, _ = precision_recall_curve(labels, probs)
    metrics = metrics_from_rows(rows)
    axes[0].plot(fpr, tpr, label=f"{name} ({metrics['roc_auc']:.4f})")
    axes[1].plot(rec, prec, label=f"{name} ({metrics['pr_auc']:.4f})")
    axes[2].hist(probs[labels == 0], bins=20, range=(0, 1), histtype="step", density=True, label=f"{name} benign")
    axes[2].hist(probs[labels == 1], bins=20, range=(0, 1), histtype="step", density=True, linestyle="--", label=f"{name} standalone")
axes[0].plot([0, 1], [0, 1], "k:", linewidth=1)
axes[0].set_title("ROC")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[1].set_title("Precision-Recall")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[2].set_title("Score Histograms")
axes[2].set_xlabel("Standalone probability")
axes[2].set_ylabel("Density")
for ax in axes:
    ax.legend(fontsize=8)
fig.tight_layout()
comparison_plot = VALIDATION_DIR / "stage_comparison_plots.png"
fig.savefig(comparison_plot, dpi=160)
plt.close(fig)
print(comparison_plot)

hw_metrics = metrics_from_rows(hw_rows)
save_checkpoint_plots(
    str(VALIDATION_DIR),
    "final",
    canonical_metrics=hw_metrics,
    split_info=f"Candidate: cnn_small_hls_opt_img512 | Fold: {SELECTED_FOLD} | Stage: U55C hardware",
    run_params={"hls_sweep": HLS_SWEEP_ROOT.name, "board": "u55c", "abi": "ap_fixed<16,6> packed AXI512"},
)
validation_manifest = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "fold": SELECTED_FOLD,
    "hls_sweep": HLS_SWEEP_ROOT.name,
    "comparison_summary": str(VALIDATION_DIR / "comparison_summary.json"),
    "comparison_plot": str(comparison_plot),
    "final_evaluation_plots": str(VALIDATION_DIR / "final_evaluation_plots.png"),
    "hardware_per_sample_enriched": str(U55C_ROOT / "hardware_per_sample_enriched.csv"),
}
write_json(VALIDATION_DIR / "validation_manifest.json", validation_manifest)
print("final evaluation plots:", VALIDATION_DIR / "final_evaluation_plots.png")
